# A polynomial in two inputs

A response surface is polynomial regression with one column for every monomial in the inputs. The product $xz$ is the bilinear term. It lets the slope in $x$ depend on $z$, and it is still a known column.


## Four corners

$$
\begin{array}{c|cc}
z \backslash x & 0 & 1 \\
\hline
0 & 1 & 3 \\
1 & 4 & 10
\end{array}
$$

Fit $b_0 + b_1 x + b_2 z + b_3 xz$ by substituting the corners, starting where both inputs vanish.

$$
\begin{aligned}
(0, 0) &\Longrightarrow b_0 = 1, \\
(1, 0) &\Longrightarrow b_0 + b_1 = 3 \Longrightarrow b_1 = 2, \\
(0, 1) &\Longrightarrow b_0 + b_2 = 4 \Longrightarrow b_2 = 3, \\
(1, 1) &\Longrightarrow 1 + 2 + 3 + b_3 = 10 \Longrightarrow b_3 = 4.
\end{aligned}
$$

The surface is $1 + 2x + 3z + 4xz$. The slope in $x$ is $2 + 4z$: it equals $2$ on the edge $z = 0$ and $6$ on the edge $z = 1$. Four coefficients and four corners give residual sum zero. This is interpolation on the corners, the two-input analogue of a polynomial whose degree uses up the sample.


In [1]:
# Substitution at the four corners recovers 1 + 2x + 3z + 4xz.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
y = np.array([1.0, 3, 4, 10])
X = np.column_stack([np.ones(4), x, z, x * z])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta", beta)
print("slope in x at z=0 and z=1", beta[1], beta[1] + beta[3])
assert np.allclose(beta, [1, 2, 3, 4])
assert np.sum((y - X @ beta) ** 2) < 1e-12


beta [1. 2. 3. 4.]
slope in x at z=0 and z=1 2.0 6.0


## Dropping the product

The additive polynomial $b_0 + b_1 x + b_2 z$ has normal equations

$$
X^{\mathsf T}X = \begin{bmatrix} 4 & 2 & 2 \\ 2 & 2 & 1 \\ 2 & 1 & 2 \end{bmatrix}, \qquad
X^{\mathsf T}y = \begin{bmatrix} 18 \\ 13 \\ 14 \end{bmatrix}.
$$

The solution is $b_0 = 0$, $b_1 = 4$, $b_2 = 5$. Predictions are $0, 4, 5, 9$, against responses $1, 3, 4, 10$, so the residuals are $1, -1, -1, 1$ and

$$
\mathrm{RSS} = 4.
$$

One slope in $x$ cannot equal both $3 - 1 = 2$ and $10 - 4 = 6$. The product column is the polynomial term that carries that change of slope. Removing it costs a residual sum of $4$ on these four corners.


In [2]:
# Without xz the fit is 4x + 5z and the residual sum of squares is 4.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
y = np.array([1.0, 3, 4, 10])
X = np.column_stack([np.ones(4), x, z])
beta = np.linalg.solve(X.T @ X, X.T @ y)
residuals = y - X @ beta
print("beta", beta)
print("residuals", residuals, "RSS", np.sum(residuals ** 2))
assert np.allclose(beta, [0, 4, 5])
assert np.allclose(residuals, [1, -1, -1, 1])
assert np.sum(residuals ** 2) == 4


beta [0. 4. 5.]
residuals [ 1. -1. -1.  1.] RSS 4.0


## One change: a square in one input

Add the column $x^2$ to a design that also contains an off-corner point, $(2, 0)$, with response $1 + 2\cdot 2 + 3\cdot 0 + 4\cdot 0 + 1\cdot 4 = 9$ from the richer law

$$
1 + 2x + 3z + 4xz + x^2.
$$

At the original corners $x$ is $0$ or $1$, so $x^2 = x$ and the square is confounded with the linear term. The new point $x = 2$ breaks that tie: $2^2 \neq 2$. Five coefficients and five points of a full-rank design recover the law exactly, including the coefficient $1$ of $x^2$. A square is identifiable only where the inputs make $x^2$ linearly independent of $1$ and $x$.


In [3]:
# The point x=2 separates x squared from x. The five-parameter law is recovered.
import numpy as np
x = np.array([0.0, 1, 0, 1, 2])
z = np.array([0.0, 0, 1, 1, 0])
y = 1 + 2 * x + 3 * z + 4 * x * z + x ** 2
X = np.column_stack([np.ones(5), x, z, x * z, x ** 2])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta", beta)
print("y", y)
assert np.allclose(beta, [1, 2, 3, 4, 1])
assert np.linalg.matrix_rank(X) == 5


beta [1. 2. 3. 4. 1.]
y [ 1.  4.  4. 11.  9.]


## Pitfall

On the four corners alone, appending $x^2$ does not create a new direction: $x^2 = x$ at $x \in \{0, 1\}$. The five-column matrix has rank $4$, and the normal equations do not have a unique solution. A printed coefficient on $x^2$ from that design would be an artifact of how a solver breaks a tie, not a measured curvature. The extra point $x = 2$ is what makes the curvature a column of its own.


In [4]:
# On the corners, x squared equals x, so the square column is redundant.
import numpy as np
x = np.array([0.0, 1, 0, 1])
z = np.array([0.0, 0, 1, 1])
X = np.column_stack([np.ones(4), x, z, x * z, x ** 2])
print("x squared minus x", x ** 2 - x)
print("rank", np.linalg.matrix_rank(X), "columns", X.shape[1])
assert np.allclose(x ** 2, x)
assert np.linalg.matrix_rank(X) == 4


x squared minus x [0. 0. 0. 0.]
rank 4 columns 5


## Summary

- The corner table is the polynomial $1 + 2x + 3z + 4xz$. The slope in $x$ is $2$ when $z = 0$ and $6$ when $z = 1$.
- Without the product, the fit is $4x + 5z$ and $\mathrm{RSS} = 4$.
- At $x \in \{0, 1\}$, the column $x^2$ duplicates $x$. An input at $x = 2$ separates them.
